In [6]:
import warnings
from IPython.core.interactiveshell import InteractiveShell
from IPython.display import display, Markdown, HTML, Javascript
from tqdm import tqdm
import geopandas as gpd
import pandas as pd


def alert(message='Loop completed!'):
    # Check if the system supports 'say' and notifications
    os.system(f'osascript -e \'display notification "{message}" with title "Notification"\'')
    # Speak the alert
    os.system(f'say "{message}"')        
        
InteractiveShell.ast_node_interactivity = "all"
warnings.filterwarnings("ignore")
tqdm.pandas()

### Define Ballotpedia API lookup & processing functions
Caches and retrieves Ballotpedia geographic data for a given latitude/longitude. Uses polite rate limiting and request headers. Function has a 100,000 item cache to avoid duplicate API calls.

Functions to transform raw API election data into a structured format:

Process ballot measures and candidate information
Calculate decision metrics (number of races and options)
Format final output as a pandas DataFrame with location data

In [7]:
import requests
import pandas as pd
from functools import lru_cache
from typing import Dict, List, Optional
from collections import defaultdict

# Define Ballotpedia API lookup
@lru_cache(maxsize=100_000)
def get_ballotpedia_data_rigorous(lat, lng, rate_limit=2):
    url = "https://api4.ballotpedia.org/myvote_redistricting_with_historical"
    params = {
        'long': str(lng),
        'lat': str(lat),
        'include_volunteer': 'true'
    }
    headers = {
        'Accept': 'application/json',
        'Content-Type': 'application/json',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36',
        'Referer': 'https://sblv3.ballotpedia.org/',
        'Origin': 'https://sblv3.ballotpedia.org'
    }
    
    response = requests.get(url, params=params, headers=headers)

    # Rate limit
    time.sleep(rate_limit)
    
    if response.json().get('message') == 'Forbidden':
        raise PermissionError


    # print(response.json())
    return response.json()


# Define processing functions
def extract_election_data(api_response: Dict) -> List[Dict]:
    return api_response.get('data', {}).get('elections', [])


#Process ballot measures and candidate information
def process_ballot_measure(measure: Dict, common_data: Dict) -> Dict:
    return {
        **common_data,
        'race_type': 'Ballot Measure',
        'office.name': measure['name'],
        'office.type': 'Ballot Measure',
        'office.level': common_data['district_type'],
        'office.branch': 'N/A',
        'number_of_seats': 1,
        'person.name': 'Yes/No Question',
        'person.url': None,
        'party_affiliation': None,
        'status': 'On the Ballot',
        'is_incumbent': False,
        'running_mate.name': None,
        'measure_id': measure['id'],
        'measure_district_type': measure['district_type']
    }

def process_candidate(candidate: Dict, race: Dict, common_data: Dict) -> Dict:
    office = race['office']
    return {
        **common_data,
        'race_type': 'Candidate',
        'office.name': office['name'],
        'office.type': office['type'],
        'office.level': office['level'],
        'office.branch': office['branch'],
        'number_of_seats': race['number_of_seats'],
        'person.name': candidate['person']['name'],
        'person.url': candidate['person']['url'],
        'party_affiliation': candidate['party_affiliation'],
        'status': candidate['status'],
        'is_incumbent': candidate['is_incumbent'],
        'running_mate.name': candidate['running_mate']['name'] if candidate.get('running_mate') else None,
        'measure_id': None,
        'measure_district_type': None
    }

def process_district(district: Dict, election_date: str) -> List[Dict]:
    common_data = {
        'election_date': election_date,
        'district_name': district['name'],
        'district_type': district['type']
    }
    
    ballot_measures = [process_ballot_measure(measure, common_data) for measure in district.get('ballot_measures') or []]
    candidates = [process_candidate(candidate, race, common_data) 
                  for race in district.get('races')  or []
                  for candidate in race['candidates']]
    
    return ballot_measures + candidates

#Calculate decision metrics (number of races and options)
def calculate_decision_metrics(df: pd.DataFrame) -> pd.DataFrame:
    def count_decisions_and_options(group):
        decisions = defaultdict(int)
        for _, row in group.iterrows():
            if row['race_type'] == 'Ballot Measure':
                decisions[row['office.name']] = 2  # Yes/No options
            else:
                decisions[row['office.name']] += 1
        
        unique_decisions = len(decisions)
        total_options = sum(decisions.values())
        
        group['unique_decisions'] = unique_decisions
        group['total_options'] = total_options
        return group

    return df.groupby(['district_name', 'district_type']).apply(count_decisions_and_options).reset_index(drop=True)

    
#Format final output as a pandas DataFrame with location data
def process_api_response(api_response: Dict, lat: float, lng: float) -> Optional[pd.DataFrame]:
    if not api_response:
        return None
        
    elections = extract_election_data(api_response)
    if not elections:
        return None
    
    processed_data = [
        item for election in elections
        for district in election['districts']
        for item in process_district(district, election['date'])
    ]
    
    df = pd.DataFrame(processed_data)
    df['group_id'] = df.groupby(['district_name', 'district_type']).ngroup()
    df['lat'], df['lng'] = lat, lng
    
    df = calculate_decision_metrics(df)
    
    return df


### Set up dataframe 

In [8]:
import json

#Read the areas.shp shapefile and find the center_lat value that becomes the 'latlng' column
area_df = gpd.read_file('data/raw/areas.shp')
area_df['latlng'] = area_df.lookup_lat.fillna(area_df.center_lat).apply(eval)
area_df = area_df.rename(columns={'intersecti': 'intersection_id'})
area_df['district'] = area_df.district.str.replace('00', '01')
#AZspecific filtering by 'state id' 
az_area_df = area_df[area_df['state_id'] == 'AZ']
az_area_df 

# Define the desired column names
column_names = ['zip', 'state', 'district', 'county', 'lat', 'lng', 'response']

#running this cell to CREATE an empty result_df 
# Create an empty DataFrame with the specified columns
result_df = pd.DataFrame(columns=column_names)


#we have no idea 
response_lookup = dict(result_df
                       .drop_duplicates(['zip', 'lat', 'lng'])
                       .set_index(['zip', 'lat', 'lng'])
                       .response.apply(json.loads))
response_lookup.update(dict(result_df
                            .drop_duplicates(['zip', 'county', 'district'])
                            .set_index(['zip', 'county', 'district'])
                            .response.apply(json.loads)))



,intersection_id,zip,county,district,population,state,state_id,center_lat,lookup_lat,best_width,link,geometry,latlng
7459,8131,85333,Yuma,09,501.0,Arizona,AZ,"(32.921822005682955, -113.43842065650553)","(32.921822005682955, -113.43842065650553)",18843.705658,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((-1598560.556 1262192.214, -1598569.3...","(32.921822005682955, -113.43842065650553)"
7460,8133,85364,Yuma,09,72225.0,Arizona,AZ,"(32.71447792615716, -114.59443219418046)","(32.71447792615716, -114.59443219418046)",2745.769737,https://edbltn.github.io/show-me-the-ballot/da...,"MULTIPOLYGON (((-1723351.431 1242984.374, -172...","(32.71447792615716, -114.59443219418046)"
7461,8134,85364,Yuma,07,72225.0,Arizona,AZ,"(32.700761056908796, -114.6790922837226)","(32.70469, -114.65664)",5737.739525,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((-1736919.04 1238304.738, -1736909.98...","(32.70469, -114.65664)"
7462,8135,85364,Yuma,25,72225.0,Arizona,AZ,"(32.73888145307751, -114.70208682910409)","(32.73888145307751, -114.70208682910409)",518.269886,https://edbltn.github.io/show-me-the-ballot/da...,"MULTIPOLYGON (((-1729618.755 1244548.468, -172...","(32.73888145307751, -114.70208682910409)"
7463,8136,85365,Yuma,09,49699.0,Arizona,AZ,"(33.08289818159969, -114.12552921403879)","(33.07067, -114.13801)",44598.032590,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((-1731183.982 1237122.048, -1731060.8...","(33.07067, -114.13801)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
48629,52859,86547,Apache,02,1040.0,Arizona,AZ,"(36.527557356440724, -109.4602850301337)","(36.52759, -109.46019)",14165.229360,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((-1203091.333 1582760.664, -1203073.6...","(36.52759, -109.46019)"
52447,57041,86514,San Juan,03,3367.0,Arizona,AZ,"(37.05063360118225, -109.19017149680822)","(37.05063360118225, -109.19017149680822)",11085.947587,https://edbltn.github.io/show-me-the-ballot/da...,"POLYGON ((-1150265.516 1629999.619, -1150659.6...","(37.05063360118225, -109.19017149680822)"
52460,57059,86044,San Juan,02,3149.0,Arizona,AZ,"(37.15872010360266, -110.92905825511951)","(37.15872010360266, -110.92905825511951)",730.892388,https://edbltn.github.io/show-me-the-ballot/da...,"MULTIPOLYGON (((-1306081.016 1672202.098, -130...","(37.15872010360266, -110.92905825511951)"
52461,57060,86044,San Juan,02,3149.0,Arizona,AZ,"(37.002713381924515, -110.87524107473315)","(37.002713381924515, -110.87524107473315)",37.699200,https://edbltn.github.io/show-me-the-ballot/da...,"MULTIPOLYGON (((-1270711.186 1647415.589, -127...","(37.002713381924515, -110.87524107473315)"


### Define main scraping and processing loop
Processes and accumulates data for each geographic point through the Ballotpedia API

In [78]:
import os
import time

# Pandas(Index=7459, intersection_id=8131, zip='85333', county='Yuma', district='09', population=501.0, state='Arizona', 
#        state_id='AZ', center_lat='(32.921822005682955, -113.43842065650553)', 
#        lookup_lat='(32.921822005682955, -113.43842065650553)', best_width=18843.70565791754, 
#        link='https://edbltn.github.io/show-me-the-ballot/data/processed/85333.html', 
#        geometry=<POLYGON ((-1598560.556 1262192.214, -1598569.343 1262144.574, -1598575.741 ...>, 
                                                                                                                                                                                                                                                                                                                                                                                                                                                          
# Define the desired column names
column_names = ['intersection_id', 'zip', 'county', 'district', 'population', 'state', 'state_id', 'center_lat', 'lookup_lat', 'best_width', 'link', 'geometry', 'latlng', 'response']

def process_dataframe(df):
    all_results = []
    
    for row in tqdm(df.itertuples(), total=len(df), desc="Processing rows"):

        lat, lng = row.latlng
        key1 = (str(int(row.zip)), str(lat)[:12], str(lng)[:12])
        key2 = (f'{int(row.zip):05}', str(lat)[:12], str(lng)[:12])
        key3 = (str(int(row.zip)), row.county, row.district)
        key4 = (str(int(row.zip)), row.county, None)
        key5 = (str(int(row.zip)), row.county, 'ne')
        key6 = (str(int(row.zip)), row.county, '00')
        key7 = (str(int(row.zip)), row.county, '01')
        for key_ in [key1, key2, key3, key4, key5, key6, key7]:
            if key_ in response_lookup:
                break
        key = key_
        if key in response_lookup:
            response = response_lookup[key]
        else:
            response = None
            for i in range(3):
                try:
                    response = get_ballotpedia_data_rigorous(lat, lng, rate_limit=0.2)
                    # if response['data']['districts'] is not None:
                    #     response['data']['districts'][0]
                    break
                except PermissionError:
                    if i == 0:
                        print('PermissionError:', lat, lng, row.state, row.county)
                    os.system('say "beep"')  # Uses system text-to-speech to make a sound
                    time.sleep(3**(i))
                    continue
                except Exception as e:
                    print('Exception:', e, lat, lng, row.state, row.county)
                    #print(response)
                    print(e)
                    time.sleep(2**(i))
                    continue
        #NOTE: we're not using the decision_metrics method here; to be used later
        response_df = process_api_response(response, lat, lng)


#for debugging: 
        # if response_df is None:
        #     display(row.zip)
        #     display(row.county)
        #     display(row.district)
        #     continue
        # else:
        #     response_lookup[row.zip, row.county, row.district] = response

#commenting the below to prevent duplication  
        # response_df['response'] = json.dumps(response)
        # response_df['intersection_id'] = row.intersection_id
        # response_df['district'] = row.district
        # response_df['county'] = row.county
        # response_df['state'] = row.state
        # response_df['zip'] = row.zip
        # response_df['geometry'] = row.geometry

#using the 'row' in pandas 
        #result_df = pd.DataFrame(columns=column_names)
        #pandas.DataFrame(data=None, index=None, columns=None, dtype=None, copy=None)[source]
        #df.loc[len(df)] = new_person

        # print(row)
        # faulty row???  start_df = pd.DataFrame(row, columns=column_names)
        start_df = pd.DataFrame(columns=column_names)

        # row.drop('index')
        start_df.loc[len(start_df)] = row
        
        #del row['Index']
        print(dir(row))
        start_df['response'] = json.dumps(response)
        
        print(start_df)
        results = start_df.to_dict(orient='records')

        # results = row
        
        all_results.extend(results)
        
    # return all_results (failed attempt 10/16)
        
    return pd.DataFrame(all_results)

Intermediate step

In [94]:
import os
import time

# Pandas(Index=7459, intersection_id=8131, zip='85333', county='Yuma', district='09', population=501.0, state='Arizona', 
#        state_id='AZ', center_lat='(32.921822005682955, -113.43842065650553)', 
#        lookup_lat='(32.921822005682955, -113.43842065650553)', best_width=18843.70565791754, 
#        link='https://edbltn.github.io/show-me-the-ballot/data/processed/85333.html', 
#        geometry=<POLYGON ((-1598560.556 1262192.214, -1598569.343 1262144.574, -1598575.741 ...>, 
                                                                                                                                                                                                                                                                                                                                                                                                                                                          
# Define the desired column names
column_names = ['intersection_id', 'zip', 'county', 'district', 'population', 'state', 'state_id', 'center_lat', 'lookup_lat', 'best_width', 'link', 'geometry', 'latlng', 'response']

def process_dataframe(df):
    all_results = []
    
    for row in tqdm(df.itertuples(), total=len(df), desc="Processing rows"):

        lat, lng = row.latlng
        for i in range(3):
            try:
                response = get_ballotpedia_data_rigorous(lat, lng, rate_limit=0.2)
                # if response['data']['districts'] is not None:
                #     response['data']['districts'][0]
                break
            except PermissionError:
                if i == 0:
                    print('PermissionError:', lat, lng, row.state, row.county)
                os.system('say "beep"')  # Uses system text-to-speech to make a sound
                time.sleep(3**(i))
                continue
            except Exception as e:
                print('Exception:', e, lat, lng, row.state, row.county)
                #print(response)
                print(e)
                time.sleep(2**(i))
                continue
        #NOTE: we're not using the decision_metrics method here; to be used later
        response_df = process_api_response(response, lat, lng)


#using the 'row' in pandas 
        #result_df = pd.DataFrame(columns=column_names)
        #pandas.DataFrame(data=None, index=None, columns=None, dtype=None, copy=None)[source]
        #df.loc[len(df)] = new_person

        # print(row)
        # faulty row???  start_df = pd.DataFrame(row, columns=column_names)

        # df = pd.DataFrame.from_records([person_data], columns=person_data._fields)
        
        # start_df = pd.DataFrame(columns=column_names)
        
        start_df = pd.DataFrame.from_records([row], columns=row._fields)
        

        # row.drop('index')
        # start_df.loc[len(start_df)] = row

        # start_df['county'] = row.county
        # start_df[0] = row.county
        
        #del row['Index']
        # print(dir(row))
        start_df['response'] = json.dumps(response)
        
        print(start_df)
        # results = start_df.to_dict(orient='records')

        # results = row
        #extend is a form of concatenation 
        all_results.extend(start_df)
        
        
    return pd.DataFrame(all_results)

CLEAN VERSION

In [101]:
import os
import time
                                                                                                                                                                                                                                                                                                                                                                                                                                                          
# Define the desired column names
column_names = ['intersection_id', 'zip', 'county', 'district', 'population', 'state', 'state_id', 'center_lat', 'lookup_lat', 'best_width', 'link', 'geometry', 'latlng', 'response']

def process_dataframe(df):
    all_results = pd.DataFrame()
    
    for row in tqdm(df.itertuples(), total=len(df), desc="Processing rows"):

        lat, lng = row.latlng
        for i in range(3):
            try:
                response = get_ballotpedia_data_rigorous(lat, lng, rate_limit=0.2)
                break
            except PermissionError:
                if i == 0:
                    print('PermissionError:', lat, lng, row.state, row.county)
                os.system('say "beep"')  # Uses system text-to-speech to make a sound
                time.sleep(3**(i))
                continue
            except Exception as e:
                print('Exception:', e, lat, lng, row.state, row.county)
                print(e)
                time.sleep(2**(i))
                continue
        #NOTE: we're not using the decision_metrics method here; to be used later
        response_df = process_api_response(response, lat, lng)
        
        row_df = pd.DataFrame.from_records([row], columns=row._fields)
        row_df['response'] = json.dumps(response)
        
        print(row_df)
        all_results.append(row_df)
        
    # return pd.DataFrame(all_results)
    return all_results

In [ ]:
#TO DO Oct 16 : brand new dataframe lololol with row.intersection, districit, county, state, zip and response for raw JSON data 

In [ ]:
### Do the scraping

In [97]:
result_df = process_dataframe(az_area_df)

Processing rows: 100%|███████████████████████████████████████| 666/666 [00:03<00:00, 201.94it/s]


In [98]:
result_df.head(10)

,0
0,Index
1,intersection_id
2,zip
3,county
4,district
5,population
6,state
7,state_id
8,center_lat
9,lookup_lat


In [62]:
result_df = process_dataframe(az_area_df)

Processing rows:   2%|▋                                        | 12/666 [00:00<00:10, 60.02it/s]

   intersection_id   zip county district population  state state_id  \
0             7459  8131  85333     Yuma         09  501.0  Arizona   

  center_lat                                 lookup_lat  \
0         AZ  (32.921822005682955, -113.43842065650553)   

                                  best_width          link  \
0  (32.921822005682955, -113.43842065650553)  18843.705658   

                                            geometry  \
0  https://edbltn.github.io/show-me-the-ballot/da...   

                                              latlng  \
0  POLYGON ((-1598560.5563676076 1262192.21407036...   

                                            response  
0  {"success": true, "data": {"districts": [{"id"...  
   intersection_id   zip county district population    state state_id  \
0             7460  8133  85364     Yuma         09  72225.0  Arizona   

  center_lat                                lookup_lat  \
0         AZ  (32.71447792615716, -114.59443219418046)   

             

Processing rows:   6%|██▏                                     | 37/666 [00:00<00:06, 101.40it/s]

   intersection_id   zip county district population    state state_id  \
0             7572  8257  85138    Pinal         02  46647.0  Arizona   

  center_lat                                lookup_lat             best_width  \
0         AZ  (33.01618677491856, -111.98642867897556)  (33.0151, -111.98681)   

          link                                           geometry  \
0  9248.028054  https://edbltn.github.io/show-me-the-ballot/da...   

                                              latlng  \
0  POLYGON ((-1484805.658357748 1220706.575020381...   

                                            response  
0  {"success": true, "data": {"districts": [{"id"...  
   intersection_id   zip county district population    state state_id  \
0             7573  8258  85123    Pinal         07  17032.0  Arizona   

  center_lat                                 lookup_lat  \
0         AZ  (32.683759924157194, -111.70123443111483)   

                                  best_width         link  \
0

Processing rows:  10%|███▉                                     | 64/666 [00:00<00:06, 92.93it/s]

...

   intersection_id    zip county  district population    state state_id  \
0            16250  17640  85015  Maricopa         03  41848.0  Arizona   

  center_lat                                lookup_lat  \
0         AZ  (33.505634283961506, -112.1020272516206)   

               best_width         link  \
0  (33.50908, -112.10184)  1994.713114   

                                            geometry  \
0  https://edbltn.github.io/show-me-the-ballot/da...   

                                              latlng  \
0  POLYGON ((-1480046.0665973416 1283526.34981233...   

                                            response  
0  {"success": true, "data": {"districts": [{"id"...  
   intersection_id    zip county  district population    state state_id  \
0            16251  17641  85017  Maricopa         08  41195.0  Arizona   

  center_lat                                lookup_lat  \
0         AZ  (33.53686017786717, -112.12132619549445)   

                                 best_width

Processing rows:  39%|███████████████▍                       | 263/666 [00:02<00:03, 131.04it/s]

   intersection_id    zip county  district population    state state_id  \
0            16278  17668  85255  Maricopa         01  41888.0  Arizona   

  center_lat                                lookup_lat            best_width  \
0         AZ  (33.68241227937502, -111.81767755049383)  (33.6824, -111.8177)   

          link                                           geometry  \
0  9236.900473  https://edbltn.github.io/show-me-the-ballot/da...   

                                              latlng  \
0  POLYGON ((-1462047.4473575128 1305228.22052033...   

                                            response  
0  {"success": true, "data": {"districts": [{"id"...  
   intersection_id    zip county  district population    state state_id  \
0            16279  17669  85304  Maricopa         08  28587.0  Arizona   

  center_lat                              lookup_lat              best_width  \
0         AZ  (33.5956525493222, -112.1781378137049)  (33.59566, -112.17815)   

         link 

Processing rows:  42%|████████████████▏                      | 277/666 [00:02<00:03, 120.66it/s]


KeyboardInterrupt: 

In [34]:
result_df.tail(30)

,0,response
9314,03,"{""success"": true, ""data"": {""districts"": [{""id""..."
9315,3149.0,"{""success"": true, ""data"": {""districts"": [{""id""..."
9316,Arizona,"{""success"": true, ""data"": {""districts"": [{""id""..."
9317,AZ,"{""success"": true, ""data"": {""districts"": [{""id""..."
9318,"(37.08954149721607, -110.74490281594383)","{""success"": true, ""data"": {""districts"": [{""id""..."
9319,"(37.08954149721607, -110.74490281594383)","{""success"": true, ""data"": {""districts"": [{""id""..."
9320,18269.919643,"{""success"": true, ""data"": {""districts"": [{""id""..."
9321,https://edbltn.github.io/show-me-the-ballot/da...,"{""success"": true, ""data"": {""districts"": [{""id""..."
9322,POLYGON ((-1271063.3975225189 1647583.68140929...,"{""success"": true, ""data"": {""districts"": [{""id""..."
9323,"(37.08954149721607, -110.74490281594383)","{""success"": true, ""data"": {""districts"": [{""id""..."


In [35]:
result_df

,0,response
0,7459,"{""success"": true, ""data"": {""districts"": [{""id""..."
1,8131,"{""success"": true, ""data"": {""districts"": [{""id""..."
2,85333,"{""success"": true, ""data"": {""districts"": [{""id""..."
3,Yuma,"{""success"": true, ""data"": {""districts"": [{""id""..."
4,09,"{""success"": true, ""data"": {""districts"": [{""id""..."
...,...,...
9319,"(37.08954149721607, -110.74490281594383)","{""success"": true, ""data"": {""districts"": [{""id""..."
9320,18269.919643,"{""success"": true, ""data"": {""districts"": [{""id""..."
9321,https://edbltn.github.io/show-me-the-ballot/da...,"{""success"": true, ""data"": {""districts"": [{""id""..."
9322,POLYGON ((-1271063.3975225189 1647583.68140929...,"{""success"": true, ""data"": {""districts"": [{""id""..."


In [ ]:
#result_df.to_csv('data/2025/ariz_oct19.csv')